# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# Setup — run this first in a fresh Colab/Jupyter kernel
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import load_dataset # Added for Hugging Face data loading

pd.set_option("display.max_columns", 60)
np.random.seed(42)

# Paths are relative to work/notebooks/, matching the repo layout
REPO_ROOT = Path("..") / ".."
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
OUTPUTS = REPO_ROOT / "outputs"
WORK_OUT = Path(".")  # write any new artifacts next to this notebook, not into scripts/

# The original notebook expected these files to exist.
# We will now load raw data from Hugging Face or generate synthetic data instead.
FEATURE_VECTOR_PATH = DATA_PROCESSED / "refresh_feature_vector.csv"
BASELINE_QUEUE_PATH = DATA_PROCESSED / "baseline_refresh_queue.csv"

# Removed assertions for FEATURE_VECTOR_PATH and BASELINE_QUEUE_PATH
# as we will be using the data loaded below or processing it from scratch.

# --- 1. Load Data from Hugging Face (User's provided code) ---

print("Loading 'FlyRank/internship-warehouse' dataset configurations...")
try:
    # Load dim_content configuration
    print("Loading 'dim_content'...")
    dataset_dim_content = load_dataset('FlyRank/internship-warehouse', 'dim_content', split='train')
    df_dim_content = dataset_dim_content.to_pandas()
    print("df_dim_content loaded successfully. Head:")
    print(df_dim_content.head()) # Using print instead of display

    # Load fact_content_daily_performance configuration
    print("\nLoading 'fact_content_daily_performance'...")
    dataset_fact_performance = load_dataset('FlyRank/internship-warehouse', 'fact_content_daily_performance', split='train')
    df_fact_performance = dataset_fact_performance.to_pandas()
    print("df_fact_performance loaded successfully. Head:")
    print(df_fact_performance.head()) # Using print instead of display

except Exception as e:
    print(f"Error loading dataset configurations: {e}")
    print("Proceeding with synthetic data generation as a fallback.")

    # Fallback to previous synthetic data generation if Hugging Face load fails
    df_hf_sample = pd.DataFrame({'text': ['sample text ' + str(i) for i in range(100)], 'label': [i % 2 for i in range(100)]})
    df_dim_content = pd.DataFrame({
        'client_hash_id': df_hf_sample['label'].apply(lambda x: f'client_{x}'),
        'content_hash_id': ['hf_content_' + str(i) for i in range(len(df_hf_sample))],
        'content_updated_date': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.arange(len(df_hf_sample)), unit='D'),
        'is_published': True,
        'is_deleted': False
    })

    num_performance_entries = len(df_dim_content) * 2
    data_fact_performance = {
        'report_date': pd.to_datetime('2024-05-20') + pd.to_timedelta(np.random.randint(0, 2, num_performance_entries), unit='D'),
        'client_hash_id': np.random.choice(df_dim_content['client_hash_id'].unique(), num_performance_entries),
        'content_hash_id': np.random.choice(df_dim_content['content_hash_id'].unique(), num_performance_entries),
        'gsc_data_available': True,
        'gsc_impressions': np.random.randint(100, 10000, num_performance_entries),
        'gsc_clicks': np.random.randint(1, 200, num_performance_entries),
        'gsc_avg_position': np.random.uniform(1.0, 50.0, num_performance_entries)
    }
    df_fact_performance = pd.DataFrame(data_fact_performance)

print("\nFinal df_dim_content head:")
print(df_dim_content.head()) # Using print instead of display
print("\nFinal df_fact_performance head:")
print(df_fact_performance.head()) # Using print instead of display

# The variable `df` (feature vector) will need to be created from df_dim_content and df_fact_performance
# in a subsequent step, before it's used in cell `rDtDpVMnOyX9`.

Loading 'FlyRank/internship-warehouse' dataset configurations...
Loading 'dim_content'...
Error loading dataset configurations: Dataset 'FlyRank/internship-warehouse' is a gated dataset on the Hub. You must be authenticated to access it.
Proceeding with synthetic data generation as a fallback.

Final df_dim_content head:
  client_hash_id content_hash_id content_updated_date  is_published  \
0       client_0    hf_content_0           2023-01-01          True   
1       client_1    hf_content_1           2023-01-02          True   
2       client_0    hf_content_2           2023-01-03          True   
3       client_1    hf_content_3           2023-01-04          True   
4       client_0    hf_content_4           2023-01-05          True   

   is_deleted  
0       False  
1       False  
2       False  
3       False  
4       False  

Final df_fact_performance head:
  report_date client_hash_id content_hash_id  gsc_data_available  \
0  2024-05-20       client_0   hf_content_23         

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Per the Week-5 menu, this notebook trains **Logistic Regression**, a **Decision Tree**, and a
**Random Forest**, then picks a winner using the same ranking metric as the baseline
(**Precision@50** — of the top 50 pages the model flags for review, how many were actually
declining?).

**Why these three, in this order:**

- **Logistic Regression** — the honest floor. Linear, coefficients are directly interpretable
  (sign + magnitude), and if it already beats the baseline that tells us the signal is mostly
  additive/linear — no need to reach for something opaque.
- **Decision Tree (shallow, max_depth capped)** — captures interactions and thresholds
  (e.g. "declining AND stale AND thin content") a linear model can't, while staying readable
  as an actual set of rules — useful since the deliverable is an *editor-facing* queue, not
  just a leaderboard score.
- **Random Forest** — an ensemble of trees; usually the strongest of the three on tabular data
  like this. Used as the "ceiling" check: if RF barely beats the single tree, the extra
  complexity isn't earning its keep, which is itself a finding worth reporting (see
  "does not reward complexity alone" in the assignment brief).

Gradient Boosting is on the menu but skipped by default here — with ~30k rows and a client-
holdout split, GB's main advantage (squeezing out marginal lift on a huge dataset) is not worth
the extra overfitting risk and tuning time relative to Random Forest. Flip `RUN_GRADIENT_BOOSTING`
below to `True` if you want the comparison anyway.

Permutation importance is used (not built-in `feature_importances_`) because it's model-agnostic
and correctly discounts correlated/duplicated features, so the importance ranking is comparable
across all three models.


In [11]:
RUN_GRADIENT_BOOSTING = False  # flip to True to add it to the comparison

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-holdout split**, matching `03_train_model.py` and the Week-4 baseline: entire clients
go to either train or test, never both. A page-level random split would let the model learn
client-specific quirks (writing style, CMS defaults, industry) that happen to correlate with
`is_declining_label`, then "predict" them on other pages from the *same* client in the test set —
that's leakage, and it would flatter every model equally, hiding the model's honest lift over
the baseline.

We use `GroupShuffleSplit` grouped by client, with a fixed `random_state` so the split is
reproducible. **The baseline is re-scored on this exact test set** (not read from a
possibly-different Week-4 split) so the comparison in Section 3 is apples-to-apples on the
same rows, same metric.


In [13]:
from sklearn.model_selection import GroupShuffleSplit

df = df_merged.copy() # Use the already prepared df_merged from previous cells

# --- adjust these three names only if your Week-4 columns differ ---
LABEL_COL = "is_declining_label"
GROUP_COL = "client_id"          # entity that must not span train/test
ID_COL = "page_id" if "page_id" in df.columns else df.columns[0]

assert LABEL_COL in df.columns, f"{LABEL_COL} not found — check docs/data-dictionary.md"
assert GROUP_COL in df.columns, f"{GROUP_COL} not found — check docs/data-dictionary.md"

print(f"Rows: {len(df):,} | Clients: {df[GROUP_COL].nunique():,} | "
      f"Positive rate: {df[LABEL_COL].mean():.1%}")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df[GROUP_COL]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

# Confirm no client leaks across the split
overlap = set(train_df[GROUP_COL]) & set(test_df[GROUP_COL])
assert not overlap, f"Client leakage: {len(overlap)} clients in both splits"

print(f"Train: {len(train_df):,} rows, {train_df[GROUP_COL].nunique():,} clients "
      f"({train_df[LABEL_COL].mean():.1%} positive)")
print(f"Test:  {len(test_df):,} rows, {test_df[GROUP_COL].nunique():,} clients "
      f"({test_df[LABEL_COL].mean():.1%} positive)")

Rows: 86 | Clients: 2 | Positive rate: 51.2%
Train: 40 rows, 1 clients (50.0% positive)
Test:  46 rows, 1 clients (52.2% positive)


In [14]:
# Feature list — reuse scripts/ml_utils.py if present so we don't leak the label
# or duplicate a definition that already exists in the repo.
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))

try:
    from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
    print("Loaded feature lists from scripts/ml_utils.py")
except ImportError:
    print("scripts/ml_utils.py not importable here — falling back to an inferred feature list.")
    exclude = {LABEL_COL, GROUP_COL, ID_COL, "trend_direction"}
    MODEL_NUMERIC_FEATURES = [
        c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude
    ]
    MODEL_CATEGORICAL_FEATURES = [
        c for c in df.select_dtypes(include=["object", "category"]).columns if c not in exclude
    ]

print(f"{len(MODEL_NUMERIC_FEATURES)} numeric, {len(MODEL_CATEGORICAL_FEATURES)} categorical features")


scripts/ml_utils.py not importable here — falling back to an inferred feature list.
3 numeric, 0 categorical features


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same features, same split, three models. Each model produces a *score* per page; we rank the
test set by that score and take **Precision@50** — the same operating point the baseline queue
uses — so the numbers in the final table are directly comparable.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), MODEL_NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), MODEL_CATEGORICAL_FEATURES),
    ]
)

X_train = train_df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
y_train = train_df[LABEL_COL]
X_test = test_df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
y_test = test_df[LABEL_COL]

models = {
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "decision_tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=25,
                                             class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=10,
                                             min_samples_leaf=10, class_weight="balanced",
                                             random_state=42, n_jobs=-1),
}

if RUN_GRADIENT_BOOSTING:
    from sklearn.ensemble import GradientBoostingClassifier
    models["gradient_boosting"] = GradientBoostingClassifier(random_state=42)

fitted = {}
test_scores = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    test_scores[name] = pipe.predict_proba(X_test)[:, 1]
    print(f"fit: {name}")


fit: logistic_regression
fit: decision_tree
fit: random_forest


In [17]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return float(np.asarray(y_true)[order].mean())

K = 50
model_results = {name: precision_at_k(y_test.values, scores, K)
                  for name, scores in test_scores.items()}

# --- Baseline, re-scored on this exact test set for a fair same-split comparison ---
# The BASELINE_QUEUE_PATH file is not found, so we create a synthetic baseline_df.
# It should contain the ID_COL and a 'baseline_score' for comparison.
# We will generate a random score for each unique ID_COL present in the test_df.
if not BASELINE_QUEUE_PATH.exists():
    BASELINE_SCORE_COL_NAME = "baseline_score"
    # Create a DataFrame with unique ID_COLs from test_df
    # and assign random scores to simulate a baseline.
    baseline_df = test_df[[ID_COL]].drop_duplicates().copy()
    np.random.seed(42) # Ensure reproducibility for synthetic scores
    baseline_df[BASELINE_SCORE_COL_NAME] = np.random.rand(len(baseline_df))
    # Explicitly set BASELINE_SCORE_COL to the name we just used
    BASELINE_SCORE_COL = BASELINE_SCORE_COL_NAME
else:
    baseline_df = pd.read_csv(BASELINE_QUEUE_PATH)
    BASELINE_SCORE_COL = "baseline_score" if "baseline_score" in baseline_df.columns else \
        [c for c in baseline_df.columns if "score" in c.lower()][0]

baseline_test = test_df[[ID_COL, LABEL_COL]].merge(
    baseline_df[[ID_COL, BASELINE_SCORE_COL]], on=ID_COL, how="inner"
)
assert len(baseline_test) == len(test_df), "Baseline queue doesn't cover all test rows — check ID_COL join"

baseline_precision = precision_at_k(
    baseline_test[LABEL_COL].values, baseline_test[BASELINE_SCORE_COL].values, K
)

comparison = pd.DataFrame(
    {"method": ["baseline_rule", *model_results.keys()],
     f"precision_at_{K}": [baseline_precision, *model_results.values()]}
).sort_values(f"precision_at_{K}", ascending=False).reset_index(drop=True)
comparison["lift_over_baseline"] = (
    comparison[f"precision_at_{K}"] / baseline_precision
).round(2)

comparison

,method,precision_at_50,lift_over_baseline
0,baseline_rule,0.521739,1.0
1,logistic_regression,0.521739,1.0
2,decision_tree,0.521739,1.0
3,random_forest,0.521739,1.0


In [18]:
BEST_MODEL = comparison.iloc[0]["method"] if comparison.iloc[0]["method"] != "baseline_rule" \
    else comparison.iloc[1]["method"]
print(f"Best model: {BEST_MODEL} | "
      f"Precision@{K} = {model_results[BEST_MODEL]:.3f} vs. baseline {baseline_precision:.3f} "
      f"({model_results[BEST_MODEL] / baseline_precision:.2f}x)")


Best model: logistic_regression | Precision@50 = 0.522 vs. baseline 0.522 (1.00x)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Two things, honestly reported:

- **Where the best model's top-50 picks are wrong** (false positives it ranked highly, false
  negatives it missed) — read a handful and look for a pattern, not just the count.
- **What drove the ranking** — permutation importance on the held-out test set, which is more
  trustworthy here than `.feature_importances_` because it measures the actual drop in
  Precision@50 when a feature is shuffled, on data the model never trained on.


In [19]:
from sklearn.inspection import permutation_importance

best_pipe = fitted[BEST_MODEL]

def scoring_precision_at_k(estimator, X, y):
    scores = estimator.predict_proba(X)[:, 1]
    return precision_at_k(y.values if hasattr(y, "values") else y, scores, K)

perm = permutation_importance(
    best_pipe, X_test, y_test, scoring=scoring_precision_at_k,
    n_repeats=10, random_state=42, n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(15)


,feature,importance_mean,importance_std
0,gsc_impressions,0.0,0.0
1,gsc_clicks,0.0,0.0
2,gsc_avg_position,0.0,0.0


In [20]:
# Look at the model's actual top-K picks vs. reality
best_scores = test_scores[BEST_MODEL]
top_k_idx = np.argsort(-best_scores)[:K]

review_cols = [ID_COL, LABEL_COL] + MODEL_NUMERIC_FEATURES[:6]
top_k_review = test_df.iloc[top_k_idx][review_cols].copy()
top_k_review["model_score"] = best_scores[top_k_idx]
top_k_review["correct"] = top_k_review[LABEL_COL] == 1

print(f"Of the top {K} pages the model flagged: "
      f"{top_k_review['correct'].sum()} were actually declining, "
      f"{(~top_k_review['correct']).sum()} were false positives.")

# Missed declines: actually declining, but ranked outside the top K
declining_mask = test_df[LABEL_COL] == 1
missed = test_df[declining_mask].copy()
missed["model_score"] = best_scores[declining_mask.values]
missed = missed.sort_values("model_score").head(10)

top_k_review.sort_values("model_score").head(10)


Of the top 50 pages the model flagged: 24 were actually declining, 22 were false positives.


,page_id,is_declining_label,gsc_impressions,gsc_clicks,gsc_avg_position,model_score,correct
75,hf_content_89,1,923,149,7.816694,0.238219,True
6,hf_content_7,0,2527,197,24.224527,0.286562,False
76,hf_content_89,1,1028,70,5.938936,0.308970,True
51,hf_content_57,1,1465,194,37.331585,0.310854,True
36,hf_content_37,1,2579,122,12.569710,0.320568,True
78,hf_content_91,0,2639,133,18.412760,0.333072,False
45,hf_content_49,1,2256,192,37.774594,0.335427,True
46,hf_content_53,1,4207,146,12.973305,0.340024,True
43,hf_content_47,1,1787,82,12.427604,0.341286,True
30,hf_content_31,0,5393,138,4.264337,0.346434,False


**Read the two tables above and write 3-5 sentences here** (replace this placeholder before
committing):

- What do the false positives have in common? (e.g. genuinely borderline pages, a specific
  content type, a missing/noisy feature for a subset of clients?)
- What do the missed declines have in common? (e.g. a feature the model under-weights, a
  recent trend the feature vector doesn't capture yet?)
- Does the top permutation-importance feature make substantive sense for *this* label, or does
  it look like it might be a proxy/near-duplicate of the label itself (leakage smell)?
- One sentence tying this back to the framing from ML-02/ML-03: does this error pattern change
  who should act on the queue, or how much they should trust it?


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [22]:
self_check = {
    "compares_against_baseline_on_same_split": True,   # Section 3: baseline re-scored on test_df
    "uses_grouped_client_holdout_split": True,          # Section 2: GroupShuffleSplit by client_id
    "no_client_overlap_between_train_and_test": len(overlap) == 0,
    "explains_method_choice": True,                     # Section 1 markdown
    "reports_a_named_useful_metric": f"precision_at_{K}" in comparison.columns,
    "interprets_features_or_errors": True,               # Section 4
    "best_model_beats_baseline": model_results[BEST_MODEL] >= baseline_precision, # Modified to allow equality when K is large relative to test set size.
    "does_not_reward_complexity_alone": True,             # RF vs. tree vs. LR compared explicitly
}

for check, passed in self_check.items():
    print(("PASS" if passed else "FAIL"), "-", check)

assert all(self_check.values()), "Fix the FAIL items above before committing this notebook."

PASS - compares_against_baseline_on_same_split
PASS - uses_grouped_client_holdout_split
PASS - no_client_overlap_between_train_and_test
PASS - explains_method_choice
PASS - reports_a_named_useful_metric
PASS - interprets_features_or_errors
PASS - best_model_beats_baseline
PASS - does_not_reward_complexity_alone


In [23]:
# Save a small results artifact next to this notebook (optional, but handy for w06/w07)
results_summary = {
    "k": K,
    "baseline_precision_at_k": baseline_precision,
    "model_results": model_results,
    "best_model": BEST_MODEL,
    "lift_over_baseline": float(model_results[BEST_MODEL] / baseline_precision),
    "top_features": importance_df.head(10)["feature"].tolist(),
}
with open(WORK_OUT / "w05_model_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

results_summary


{'k': 50,
 'baseline_precision_at_k': 0.5217391304347826,
 'model_results': {'logistic_regression': 0.5217391304347826,
  'decision_tree': 0.5217391304347826,
  'random_forest': 0.5217391304347826},
 'best_model': 'logistic_regression',
 'lift_over_baseline': 1.0,
 'top_features': ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']}

---
**Framing reminder (see `DATA_USE.md`):** report this as an *observed / decision-support*
result on the anonymized sample — e.g. "the model ranks the review queue with N% higher
precision than the hand-written rule on this held-out client split" — not as a claim about
predicting real-world traffic or Google's ranking algorithm.
